# Imports, Vars and Functions

In [0]:
%pip install lightgbm catboost optuna

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px # one-liner charts, high-level
import plotly.graph_objects as go # full control chart

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool

import optuna

## Vars

In [0]:
TARGET = "Exited"
N_SPLITS = 5

categorical_columns = ["Geography", "Gender"]
numerial_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
]

drop_columns  = ["id", "CustomerId", "source"]
submission_columns = ["id", "Exited"]

## Functions

### Load Data

In [0]:
def load_data(option="local"):

  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Bank Churn/train.csv')
    test_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Bank Churn/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

### Data splitter

In [0]:
def data_spliter(df):
    train = df[df['source'] == 'train'].drop(columns=drop_columns)
    submission_df = df[df['source'] == 'test'].drop(columns=['source', TARGET])

    X_train = train_df.drop(columns=[TARGET] + drop_columns)
    y_train = train_df[TARGET]

    return X_train, y_train, submission_df

In [0]:
train_df, test_df, df = load_data(option="local")

# EDA

In [0]:
len(df)

-   **Customer** ID: Уникальный идентификатор каждого клиента.
-   **Surname**: Фамилия клиента.
-   **Credit Score**: Числовое значение, представляющее кредитный рейтинг клиента.
-   **Geography**: Страна проживания клиента (Франция, Испания или Германия).
-   **Gender**: Пол клиента (Мужской или Женский).
-   **Age**: Возраст клиента.
-   **Tenure**: Количество лет, которое клиент обслуживается в банке.
-   **Balance**: Баланс на счёте клиента.
-   **NumOfProducts**: Количество банковских продуктов, которыми пользуется клиент (например, сберегательный счёт, кредитная карта).
-   **HasCrCard**: Наличие кредитной карты у клиента (1 = да, 0 = нет).
-   **IsActiveMember**: Является ли клиент активным членом банка (1 = да, 0 = нет).
-   **EstimatedSalary**: Предполагаемая заработная плата клиента.
-   **Exited**: Ушёл ли клиент (1 = да, 0 = нет).


In [0]:
df.info()

In [0]:
df.describe().T

In [0]:
df.isna().sum()

In [0]:
df.head()

## Pairplot

In [0]:
fig = sns.pairplot(
    data=train_df[numerial_columns + [TARGET]],
    hue=TARGET,
    diag_kind="kde",
    plot_kws={"alpha": 0.5, "s": 15, "edgecolor": None},
    diag_kws={"fill": True, "alpha": 0.6},
    palette={0: "#2196F3", 1: "#FF5722"},
    # corner=True,
)
fig.figure.suptitle("Pairplot of Numerical Features by Churn Status", y=1.02, fontsize=16, fontweight="bold")

## Correlations

In [0]:
corr_df = pd.get_dummies(
        data=df.drop(columns=drop_columns + ["source"] + ["Surname"]),
        columns=["Gender", "Geography"],
    ).corr()
sorted_columns = corr_df[TARGET].abs().sort_values(ascending=False).index.tolist()
corr_df = corr_df.loc[sorted_columns, sorted_columns]
corr_mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)

plt.figure(figsize=(10,8))
sns.heatmap(
    data=corr_df,
    mask=corr_mask,
    annot=True,
    fmt='.2f',
    cmap="coolwarm",
    square=True,
    center=0
)

## Surnames

In [0]:
display(df['Surname'].value_counts().reset_index())

In [0]:
df[df['Surname'].str.endswith('ov') | df['Surname'].str.endswith('ova') | df['Surname'].str.endswith('ev') | df['Surname'].str.endswith('eva')]

In [0]:
df[df['Surname'].str.contains("?", regex=False)]

In [0]:
display(df[df['Surname'].str.contains("'", regex=False)])

## Balance

In [0]:
plt.figure(figsize=(16,9))
sns.histplot(
    data=train_df,
    x="Balance",
    hue='Exited',
    bins=50,
    multiple='fill',
)

In [0]:
df['Balance'].drop_duplicates().sort_values(ascending=False).head()

In [0]:
df[df['Balance'] > 200000]

## Salary

In [0]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=train_df,
    x="EstimatedSalary",
    hue='Exited',
    bins=50,
    multiple='fill',
)

## Age

In [0]:
px.line(x=["a","b","c"], y=[1,3,2], title="sample figure").show()

In [0]:
px.histogram(
    data_frame=df,
    x = 'Age',
    # y='Balance',
    color='Exited',
    barmode='overlay',
    opacity=0.6,
    marginal="box",
    # histfunc="avg"
    # nbins=70,
    # text_auto=True
    # range_x=[17,74]
).update_layout(bargap=0.1)

In [0]:
df['Age'].value_counts().reset_index().sort_values(by='Age').head()

In [0]:
plt.figure(figsize=(12,5))
ax = sns.countplot(
    data=train_df,
    x=df['Age'].astype(int),
    hue='Exited',
)

In [0]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=train_df,
    x="Age",
    hue='Exited',
    bins=10,
    # multiple='fill',
)
# plt.xticks(np.arange(18,75))
plt.tight_layout()

In [0]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

tree = DecisionTreeClassifier(max_leaf_nodes=7, min_samples_leaf=100, random_state=0)
tree.fit(train_df[['Age']], train_df['Exited'])

cuts = np.sort(tree.tree_.threshold[tree.tree_.threshold != -2])
print(cuts)   # e.g. [34.5, 40.5, 45.5, ...]

# train_df['age_bin'] = pd.cut(train_df['Age'], bins=[-np.inf, *cuts, np.inf])
# train_df.groupby('age_bin', observed=True)['Exited'].agg(['mean', 'size'])

In [0]:
df.groupby('Age', observed=True)['Exited'].agg(['mean', 'size'])

## Credit Score

In [0]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=train_df,
    x="CreditScore",
    hue='Exited',
    bins=5,
    multiple='fill',
)
# plt.xticks(np.arange(18,75))
plt.tight_layout()

# Feature Engineering

In [0]:
def feature_engineering(df):
  original_columns = df.columns.tolist()
  print("Engineering features...")

  # 1. Interaction & Ratio Features
  df['Balance_to_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
  df['Age_to_Tenure_Ratio'] = df['Tenure'] / (df['Age'] + 1)
  df['Products_per_Tenure'] = df['NumOfProducts'] / (df['Tenure'] + 1)

  # 2. Financial Status & Behavior Indicators
  df['Is_Zero_Balance'] = (df['Balance'] == 0).astype(int)

  # # CreditScore Binning
  # score_bins = [0, 579, 669, 739, 799, 850]
  # score_labels = ['Very_Poor', 'Fair', 'Good', 'Very_Good', 'Exceptional']
  # df['CreditScore_Segment'] = pd.cut(df['CreditScore'], bins=score_bins, labels=score_labels).astype(str)

  # # Age Binning (targeting the peak churn between 45 and 60)
  # age_bins = [0, 30, 45, 60, 100]
  # age_labels = ['Under_30', '30_to_45', '45_to_60', 'Over_60']
  # df['Age_Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels).astype(str)

  # 3. Engagement Score
  df['Customer_Activity_Index'] = df['HasCrCard'] + df['IsActiveMember'] + (df['NumOfProducts'] > 1).astype(int)

  # 4. Surname & Demographic Patterns
  # Surname length and special character rules
  df['Surname_Has_Question'] = df['Surname'].str.contains('?', regex=False).astype(int)

  # Endings often representing specific ethnicities/families in synthetic datasets
  df['Surname_Ends_With_Slavic'] = df['Surname'].str.endswith(('ov', 'ova', 'ev', 'eva')).astype(int)

  engineered_features = [k for k in df.columns.tolist() if k not in original_columns]

  print(f"\nAdded {len(engineered_features)} features")
  print(f"Features added:")
  for idx, f in enumerate(engineered_features):
    print(f"{idx + 1}: {f}")
  return df

In [0]:
df = feature_engineering(df)

In [0]:
with pd.option_context('display.max_columns', None):
    display(df)

# Modelling

In [0]:
df

In [0]:
X_train, y_train, submission_df = data_spliter(df)

In [0]:
cv = StratifiedKFold(
    n_splits = N_SPLITS,
    shuffle = True,
    random_state=42
)

In [0]:
X_train